In [23]:
import pandas as pd
import numpy as np

import mosmapapi

In [24]:
# !pip install openpyxl

In [25]:
df = pd.read_excel("../data/moscow_transformed.xlsx")
df

,ID на сайте,Источник,Название,Цена,Дата,Тип автора,Метро/Район,Адрес,lat,lng,URL,Ссылки на картинки,"Расстояние до метро, км",Этаж,Этажность здания,Вид объекта,Общая площадь
0,320731003,cian.ru,"Офис в Москва Шипиловская ул., 58к1 (25 м²)",38800,2025-08-23 02:23:27,Агентство,Шипиловская,"Шипиловская ул., 58к1",55.621211,37.745600,https://www.cian.ru/rent/commercial/320731003,['https://images.cdn-cian.ru/images/2592583348...,0.083,4.0,6.0,Офисное помещение,25.0
1,321876001,cian.ru,"Офис в Москва Ленинградский просп., 47С2 (25 м²)",31250,2025-09-25 20:21:36,Агентство,Аэропорт,"Ленинградский просп., 47С2",55.799036,37.532601,https://www.cian.ru/rent/commercial/321876001,['https://images.cdn-cian.ru/images/2631122267...,0.250,-2.0,7.0,Офисное помещение,25.0
2,322186790,cian.ru,"Офис в Москва Волгоградский просп., 2 (20 м²)",29200,2025-09-25 10:22:06,Агентство,Пролетарская,"Волгоградский просп., 2",55.731176,37.669279,https://www.cian.ru/rent/commercial/322186790,['https://images.cdn-cian.ru/images/2641248288...,0.250,6.0,16.0,Офисное помещение,20.0
3,321392086,cian.ru,Помещение свободного назначения в Москва ул. А...,32000,2025-09-02 04:23:10,Частное лицо,Бунинская Аллея,"ул. Адмирала Руднева, 20",55.540409,37.516153,https://www.cian.ru/rent/commercial/321392086,['https://images.cdn-cian.ru/images/nezhiloe-p...,0.250,3.0,7.0,Торговое / Свободного назначения,32.0
4,324943206,cian.ru,"Офис в Москва просп. Мира, 95С1 (24 м²)",44000,2025-12-11 16:28:22,Агентство,Алексеевская,"просп. Мира, 95С1",55.808002,37.635853,https://www.cian.ru/rent/commercial/324943206,['https://images.cdn-cian.ru/images/ofis-moskv...,0.167,14.0,17.0,Офисное помещение,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9065,3987628968735107072,realty.yandex.ru,Офис (1200 м²),6000000,2025-09-05 16:00:52,Агентство,Шелепиха,"1-й Магистральный тупик, 5А",55.766132,37.531956,https://realty.ya.ru/offer/3987628968735107205/,NaN,11.917,4.0,8.0,Офисное помещение,1200.0
9066,322424654,cian.ru,"Офис в Москва Сколковское ш., вл43 (1149 м²)",6894000,2025-10-03 08:21:53,Агентство,Кунцевская,"Сколковское ш., вл43",55.699654,37.397701,https://www.cian.ru/rent/commercial/322424654,['https://images.cdn-cian.ru/images/2650029058...,7.333,1.0,6.0,Офисное помещение,1149.0
9067,7523548143730329600,realty.yandex.ru,Офис (967 м²),6769000,2025-10-22 07:59:36,Агентство,Немчиновка,"Сколковское шоссе, вл43",55.699654,37.397700,https://realty.ya.ru/offer/7523548143730329381/,['https://avatars.mds.yandex.net/get-realty-of...,18.333,3.0,6.0,Офисное помещение,967.0
9068,322670014,cian.ru,"Офис в Москва Сколковское ш., вл43 (1134 м²)",7938000,2025-10-11 14:21:23,Агентство,Кунцевская,"Сколковское ш., вл43",55.699654,37.397701,https://www.cian.ru/rent/commercial/322670014,['https://images.cdn-cian.ru/images/2658570375...,7.333,2.0,6.0,Офисное помещение,1134.0


In [ ]:
new_df = pd.read_csv("../data/moscow_super_transformed.csv")
if (len(new_df) < 10):
    new_df = pd.DataFrame()

def get_batch_data(l: int, r: int, radius: int): # [l, r)
    global new_df
    ls = []
    for i in range(l, r):
        if i <= new_df.shape[0]:
            continue
        lat = df.iloc[i]['lat']
        lng = df.iloc[i]['lng']
        _ =pd.DataFrame(mosmapapi.get_data_radius(lat, lng,radius), index=[0])
        _['ID на сайте'] = df.iloc[i]['ID на сайте']
        _['Источник'] = df.iloc[i]['Источник']
        ls.append(_)
    if len(ls) == 0:
        return None
    return pd.concat(ls).set_index(['ID на сайте', 'Источник'])

def get_all_data(batch_size: int, sz: int):
    global new_df
    for i in range(0, sz, batch_size):
        dt1 = get_batch_data(i, i + batch_size, 300)
        dt2 = get_batch_data(i, i + batch_size, 600)
        if dt1 is None or dt2 is None:
            continue
        _ = pd.concat([dt1, dt2.drop(
            ['district_name', 'district_price', 
             'district_price_room1', 'district_price_room2', 
             'district_price_room3', 'district_price_room4'
             ], axis=1)], axis=1)
        new_df = pd.concat([new_df, _], axis=0)
        new_df.to_csv("../data/moscow_super_transformed.csv")

In [48]:
new_df

""


In [49]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame


In [50]:
get_all_data(10, 100)

InvalidIndexError: Reindexing only valid with uniquely valued Index objects